## Общий алгоритм работы с Optuina

In [1]:
!pip install optuna -q

In [2]:
import optuna

1. Определяем целевую функцию objective, через аргументы она будет получать специальный объект trial. С его помощью можно назначать различные гипермараметры, Например, как в примере ниже, мы задаем x в интервале [-10,10].

2. Далее создаем объект обучения с помощью метода optuna.create_study.

3. Запускаем оптимизацию целевой функции objective на 10 итераций n_trials=10. Происходит 10 вызовов нашей функции с различными параметрами от -10 до 10. Какие именно параметры выбирает optuna будет описано ниже.

In [6]:
# Рассмотрим пример - будем оптимизировать функцию (x-2)**2 и будем искать ее минимум

def objective(trial):
    x = trial.suggest_float('x', -10, 10) # гиперпараметр и его границы. Также есть suggest_int, suggest_objective
    return (x - 2) ** 2 # ф-ия, которую мы минимизируем (по умолчанию)

study = optuna.create_study()
study.optimize(objective, n_trials=40) #n_trials - сколько значений гиперпараметров мы переберем

study.best_params

[I 2026-03-16 17:47:10,242] A new study created in memory with name: no-name-a015d168-6037-41d9-b871-644ba8321c95
[I 2026-03-16 17:47:10,244] Trial 0 finished with value: 2.5180594436096126 and parameters: {'x': 0.4131605488866832}. Best is trial 0 with value: 2.5180594436096126.
[I 2026-03-16 17:47:10,245] Trial 1 finished with value: 17.36451818416915 and parameters: {'x': 6.1670754953767215}. Best is trial 0 with value: 2.5180594436096126.
[I 2026-03-16 17:47:10,246] Trial 2 finished with value: 1.307932458430471 and parameters: {'x': 0.8563512521624173}. Best is trial 2 with value: 1.307932458430471.
[I 2026-03-16 17:47:10,247] Trial 3 finished with value: 59.50098877208927 and parameters: {'x': -5.7136884025794865}. Best is trial 2 with value: 1.307932458430471.
[I 2026-03-16 17:47:10,248] Trial 4 finished with value: 43.356672905830806 and parameters: {'x': 8.584578415193398}. Best is trial 2 with value: 1.307932458430471.
[I 2026-03-16 17:47:10,251] Trial 5 finished with value: 

{'x': 2.0649352701934163}

## Загрузка данных и импорт библиотек

In [8]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score

from sklearn.datasets import fetch_california_housing

In [9]:
RANDOM_STATE = 42

In [10]:
from lightgbm import LGBMRegressor

In [26]:
import sklearn

In [12]:
data = fetch_california_housing(as_frame = True)

X = data.data
y = data.target

In [13]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

## Подбор гиперпараметров с Optuna

Разобъем данные на тренировочную и тестовую часть. На тренировочной части по кросс-валидации подберем гиперпараметры моделей, а затем проверим качество на тестовой части.

In [19]:
def objective_LGBM(trial):
    max_depth = trial.suggest_int("max_depth", 2, 20)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log = True)
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)

    score = cross_val_score(LGBMRegressor(max_depth=max_depth, 
                                          learning_rate=learning_rate, 
                                          n_estimators=n_estimators),
                           Xtrain, ytrain, cv = 3, scoring = 'r2', n_jobs = -1).mean()
    return score

In [20]:
study = optuna.create_study(direction = 'maximize')
study.optimize(objective_LGBM, n_trials = 30)

[I 2026-03-16 18:00:38,503] A new study created in memory with name: no-name-50e8161c-77cf-48db-b1ee-a7f8651aaced
[I 2026-03-16 18:00:41,423] Trial 0 finished with value: 0.6751618081673113 and parameters: {'max_depth': 2, 'learning_rate': 0.021189583485515827, 'n_estimators': 205}. Best is trial 0 with value: 0.6751618081673113.
[I 2026-03-16 18:00:44,145] Trial 1 finished with value: 0.19488862259839435 and parameters: {'max_depth': 19, 'learning_rate': 0.0007982760165780928, 'n_estimators': 222}. Best is trial 0 with value: 0.6751618081673113.
[I 2026-03-16 18:00:46,548] Trial 2 finished with value: 0.17361088092719454 and parameters: {'max_depth': 8, 'learning_rate': 0.002815710604469768, 'n_estimators': 55}. Best is trial 0 with value: 0.6751618081673113.
[I 2026-03-16 18:00:49,238] Trial 3 finished with value: 0.6567911742331384 and parameters: {'max_depth': 11, 'learning_rate': 0.005690452288505957, 'n_estimators': 214}. Best is trial 0 with value: 0.6751618081673113.
[I 2026-03

In [21]:
study.best_params

{'max_depth': 17, 'learning_rate': 0.04757461530724777, 'n_estimators': 962}

In [22]:
model = LGBMRegressor(**study.best_params) # study.best_params - это словарь. Ставим **, чтобы распаковать словарь
model.fit(Xtrain, ytrain)

pred = model.predict(Xtest)

r2_score(ytest, pred)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000326 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 8
[LightGBM] [Info] Start training from score 2.070349


0.8561878606965129

Сравним результат с простым обучением с параметрами по умолчанию

In [23]:
model = LGBMRegressor() 
model.fit(Xtrain, ytrain)

pred = model.predict(Xtest)

r2_score(ytest, pred)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000343 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 8
[LightGBM] [Info] Start training from score 2.070349


0.8388070733850864

В Optuna встроена гибкая возможность перебора гиперпараметров.

 **1. Define an objective function to be maximized.**

def objective(trial):

**2. Suggest values for the hyperparameters using a trial object.**
    
    classifier_name = trial.suggest_categorical('classifier', ['SVC', 'RandomForest'])
    
    if classifier_name == 'SVC':
    
         svc_c = trial.suggest_float('svc_c', 1e-10, 1e10, log=True)
         
         classifier_obj = sklearn.svm.SVC(C=svc_c, gamma='auto')
         
    else:
    
        rf_max_depth = trial.suggest_int('rf_max_depth', 2, 32, log=True)
        
        classifier_obj = sklearn.ensemble.RandomForestClassifier(max_depth=rf_max_depth, n_estimators=10)
    ...
    return accuracy

**3. Create a study object and optimize the objective function.**

study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=100)